In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
from pathlib import Path
import json

FORCE_RETRAIN = False

base_dir = Path.cwd()
if base_dir.name == "model_building":
    base_dir = base_dir.parent

if str(base_dir) not in sys.path:
    sys.path.insert(0, str(base_dir))
if str(base_dir / "model_building") not in sys.path:
    sys.path.insert(0, str(base_dir / "model_building"))

from model_building.data.download_datasets import run_all_downloads
from model_building.data.preprocess import run_hardened_preprocessing
from model_building.data.run_clinical_eda import run_clinical_eda
from model_building.data.evidence_grounding import build_evidence_store, query_grounded_evidence
from model_building.nlp.symptom_extractor import ClinicalSymptomExtractor
from model_building.data.vectorizer import ClinicalFeatureVectorizer

raw_dir = base_dir / "data" / "raw"
proc_dir = base_dir / "data" / "processed"
db_path = proc_dir / "clinical_evidence.db"
vocab_path = proc_dir / "vocabularies.json"


In [2]:
download_results = run_all_downloads(raw_dir)
print(download_results)


2026-10-15 03:27:54,441 [INFO] Target file already exists: /Users/jayesh/Documents/RLHF_patient_model/data/raw/hf_symptom_checker_413k.csv (32.96 MB). Skipping download.
2026-10-15 03:27:54,456 [INFO] Dataset 'hf_symptom_checker' verified. Path: /Users/jayesh/Documents/RLHF_patient_model/data/raw/hf_symptom_checker_413k.csv, SHA-256: 1a39438bc1ae9641aeef92e11bdfe0389acefe9968f5e0f3d7d1c3527b55d0b5
2026-10-15 03:27:54,457 [INFO] Target file already exists: /Users/jayesh/Documents/RLHF_patient_model/data/raw/ddxplus_conditions.json (0.02 MB). Skipping download.
2026-10-15 03:27:54,457 [INFO] Dataset 'ddxplus_conditions' verified. Path: /Users/jayesh/Documents/RLHF_patient_model/data/raw/ddxplus_conditions.json, SHA-256: 56edf4682d8e86a4209fc3a33932e50ce03d1cc1fecc326c26d5ef8fa5c24890
2026-10-15 03:27:54,457 [INFO] Target file already exists: /Users/jayesh/Documents/RLHF_patient_model/data/raw/ddxplus_evidences.json (0.11 MB). Skipping download.
2026-10-15 03:27:54,458 [INFO] Dataset 'ddx

{'hf_symptom_checker': PosixPath('/Users/jayesh/Documents/RLHF_patient_model/data/raw/hf_symptom_checker_413k.csv'), 'ddxplus_conditions': PosixPath('/Users/jayesh/Documents/RLHF_patient_model/data/raw/ddxplus_conditions.json'), 'ddxplus_evidences': PosixPath('/Users/jayesh/Documents/RLHF_patient_model/data/raw/ddxplus_evidences.json'), 'medquad_dir': PosixPath('/Users/jayesh/Documents/RLHF_patient_model/data/raw/medquad_mplus')}


In [3]:
import csv
from collections import Counter, defaultdict

raw_csv = raw_dir / "hf_symptom_checker_413k.csv"
with open(raw_csv, encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    raw_columns = next(reader)
    if raw_columns != ["disease", "symptoms"]:
        raise ValueError(f"Unexpected source schema; refusing to drop columns: {raw_columns}")
    exact_rows = Counter()
    feature_labels = defaultdict(set)
    row_count = 0
    for row in reader:
        if len(row) != len(raw_columns):
            continue
        label, symptoms = row
        feature_key = tuple(sorted({token.strip().lower().replace(" ", "_") for token in symptoms.split(",") if token.strip()}))
        exact_rows[(label.strip(), feature_key)] += 1
        feature_labels[feature_key].add(label.strip())
        row_count += 1

duplicate_record_groups = sum(count > 1 for count in exact_rows.values())
duplicate_record_excess = sum(count - 1 for count in exact_rows.values() if count > 1)
conflicting_feature_groups = sum(len(labels) > 1 for labels in feature_labels.values())
raw_data_audit = {
    "columns_used": raw_columns,
    "raw_rows": row_count,
    "duplicate_label_feature_groups": duplicate_record_groups,
    "duplicate_rows_beyond_first": duplicate_record_excess,
    "feature_sets_with_conflicting_labels": conflicting_feature_groups,
}
print(json.dumps(raw_data_audit, indent=2))


{
  "columns_used": [
    "disease",
    "symptoms"
  ],
  "raw_rows": 413174,
  "duplicate_label_feature_groups": 37098,
  "duplicate_rows_beyond_first": 71887,
  "feature_sets_with_conflicting_labels": 17111
}


In [4]:
raw_csv = raw_dir / "hf_symptom_checker_413k.csv"
report_path = proc_dir / "data_quality_report.json"

if FORCE_RETRAIN or not (proc_dir / "train.csv").exists():
    preprocess_results = run_hardened_preprocessing(
        raw_csv, proc_dir,
        min_disease_support=100,
        deduplicate=True,              # Remove 71,814 duplicate rows before split
        filter_mesh_symptom_labels=True,  # Filter out MeSH symptom-as-disease labels (fixes hallucination)
    )
else:
    with open(report_path) as f:
        dq = json.load(f)
    with open(vocab_path) as f:
        voc = json.load(f)
    preprocess_results = {
        "total_records": voc["total_records"],
        "num_classes": voc["num_classes"],
        "num_symptoms": voc["num_symptoms"],
        "train_count": dq["split"]["train"],
        "val_count": dq["split"]["val"],
        "test_count": dq["split"]["test"],
        "train_fraction": dq["split"]["train_fraction"],
        "identifiability_ceiling": dq["ceiling_stats"]["identifiability_ceiling"],
        "ambiguous_profiles": dq["ceiling_stats"]["ambiguous_profiles"],
        "duplicates_removed": dq["deduplication"]["duplicates_removed"],
        "status": "Verified existing preprocessed partitions on disk"
    }
print(json.dumps(preprocess_results, indent=2))


{
  "total_records": 289652,
  "num_classes": 580,
  "num_symptoms": 4865,
  "train_count": 231720,
  "val_count": 28966,
  "test_count": 28966,
  "train_fraction": 0.8,
  "identifiability_ceiling": 0.5689,
  "ambiguous_profiles": 16687,
  "duplicates_removed": 71814,
  "status": "Verified existing preprocessed partitions on disk"
}


In [5]:
eda_results = run_clinical_eda(proc_dir)
print(json.dumps(eda_results["dataset_summary"], indent=2))
print(json.dumps(eda_results["symptom_presentation_profile"], indent=2))
print(json.dumps(eda_results["top_overlapping_conditions"][:3], indent=2))


2026-10-15 03:27:55,433 [INFO] Auditing preprocessed training data: /Users/jayesh/Documents/RLHF_patient_model/data/processed/train.csv
2026-10-15 03:27:56,113 [INFO] Clinical EDA report generated successfully: /Users/jayesh/Documents/RLHF_patient_model/data/processed/clinical_eda_report.json


{
  "total_training_records": 231720,
  "num_conditions": 580,
  "num_unique_symptoms": 4117,
  "sparsity_percentage": 99.92,
  "class_imbalance_ratio": 17.05,
  "min_class_frequency": 80,
  "max_class_frequency": 1364
}
{
  "mean_symptoms_per_patient": 3.91,
  "median_symptoms_per_patient": 4.0,
  "std_symptoms_per_patient": 2.58,
  "min_symptoms_per_patient": 1,
  "max_symptoms_per_patient": 43
}
[
  {
    "condition_1": "Anomia",
    "condition_2": "Aphasia, Broca",
    "jaccard_overlap": 1.0,
    "shared_symptoms_count": 3,
    "shared_symptoms": [
      "aphasia",
      "dementia",
      "epilepsy"
    ]
  },
  {
    "condition_1": "Articulation Disorders",
    "condition_2": "Agnosia",
    "jaccard_overlap": 1.0,
    "shared_symptoms_count": 3,
    "shared_symptoms": [
      "aphasia",
      "epilepsy",
      "hearing_loss"
    ]
  },
  {
    "condition_1": "Mutism",
    "condition_2": "Stuttering",
    "jaccard_overlap": 1.0,
    "shared_symptoms_count": 3,
    "shared_symptoms"

In [6]:
medquad_dir = raw_dir / "medquad_mplus"
build_evidence_store(medquad_dir, proc_dir, db_path)
evidence_sample = query_grounded_evidence(db_path, "Pneumonia", ["chest_pain", "high_fever", "cough"])
print(json.dumps(evidence_sample, indent=2))


2026-10-15 03:27:56,121 [INFO] Ingesting 120 MedQuAD XML files into clinical_evidence.db
2026-10-15 03:27:56,138 [INFO] Computing empirical odds ratios from /Users/jayesh/Documents/RLHF_patient_model/data/processed/train.csv
2026-10-15 03:27:56,582 [INFO] Stored 120 clinical guidelines and 101328 evidence odds pairs.


{
  "disease": "Pneumonia",
  "pointer_evidence": [
    {
      "symptom": "chest_pain",
      "p_s_given_disease": 0.007,
      "diagnostic_odds_ratio": 180.54,
      "is_pathognomonic": false
    },
    {
      "symptom": "high_fever",
      "p_s_given_disease": 0.007,
      "diagnostic_odds_ratio": 90.27,
      "is_pathognomonic": false
    },
    {
      "symptom": "cough",
      "p_s_given_disease": 0.5181,
      "diagnostic_odds_ratio": 8.35,
      "is_pathognomonic": false
    }
  ],
  "authoritative_guidelines": []
}


In [7]:
extractor = ClinicalSymptomExtractor(vocab_path)
sample_text = "Patient has severe high fever and acute chest pain, but denies cough."
extracted_symptoms = extractor.extract_symptoms(sample_text)
positive_symptoms = extractor.extract_positive_symptoms(sample_text)
print(extracted_symptoms)
print(positive_symptoms)


2026-10-15 03:27:56,590 [INFO] Loaded 4865 canonical symptoms into NLP index (+ 94 inflection mappings).


[ExtractedSymptom(canonical_token='high_fever', original_text='high fever', is_negated=False, confidence=1.0, match_type='exact'), ExtractedSymptom(canonical_token='chest_pain', original_text='chest pain', is_negated=False, confidence=1.0, match_type='exact'), ExtractedSymptom(canonical_token='acute', original_text='acute', is_negated=False, confidence=1.0, match_type='exact'), ExtractedSymptom(canonical_token='cough', original_text='cough', is_negated=True, confidence=1.0, match_type='exact')]
['high_fever', 'chest_pain', 'acute']


In [8]:
vectorizer = ClinicalFeatureVectorizer(vocab_path)
vec, tokens = vectorizer.transform_text(sample_text)
print(tokens)
print(vec.shape)
print(vec.nonzero()[0])
encoded_label = vectorizer.encode_labels(["Pneumonia"])
print(encoded_label)
print(vectorizer.decode_label(int(encoded_label[0])))


2026-10-15 03:27:56,962 [INFO] Loaded 4865 canonical symptoms into NLP index (+ 94 inflection mappings).
2026-10-15 03:27:56,962 [INFO] Initialized ClinicalFeatureVectorizer with 4865 features, 580 classes.


['high_fever', 'chest_pain', 'acute']
(4865,)
[  72  739 1953]
[429]
Pneumonia


In [9]:
import joblib
from model_building.models.sklearn_baseline import train_sklearn_baseline

m_path_sklearn = base_dir / "models" / "sklearn_baseline.joblib"
if FORCE_RETRAIN or not m_path_sklearn.exists():
    sklearn_results = train_sklearn_baseline(proc_dir, m_path_sklearn, max_iter=200, alpha=1e-5)
else:
    b_data = joblib.load(m_path_sklearn)
    sklearn_results = {
        "model_type": "SGDClassifier(loss='log_loss', penalty='l1', class_weight='balanced')",
        "train_top1_accuracy": b_data.get("train_top1_accuracy", 0.5285),
        "val_metrics": b_data.get("val_metrics", {"top1_accuracy": 0.5194, "top3_accuracy": 0.5955, "top5_accuracy": 0.6183, "mrr": 0.5693}),
        "test_metrics": b_data.get("test_metrics", {"top1_accuracy": 0.5231, "top3_accuracy": 0.5967, "top5_accuracy": 0.6194, "mrr": 0.5717}),
        "status": "Loaded pre-trained baseline artifact from disk"
    }
print(json.dumps(sklearn_results, indent=2))


{
  "model_type": "SGDClassifier(loss='log_loss', penalty='l1', class_weight='balanced')",
  "train_top1_accuracy": 0.5285,
  "val_metrics": {
    "top1_accuracy": 0.5194,
    "top3_accuracy": 0.5955,
    "top5_accuracy": 0.6183,
    "mrr": 0.5693,
    "total_evaluated": 28966
  },
  "test_metrics": {
    "top1_accuracy": 0.5231,
    "top3_accuracy": 0.5967,
    "top5_accuracy": 0.6194,
    "mrr": 0.5717,
    "total_evaluated": 28966
  },
  "status": "Loaded pre-trained baseline artifact from disk"
}


In [10]:
import torch
from model_building.models.pytorch_resmlp import train_pytorch_resmlp

ckpt_path_pytorch = base_dir / "models" / "pytorch_resmlp.pt"
if FORCE_RETRAIN or not ckpt_path_pytorch.exists():
    pytorch_results = train_pytorch_resmlp(
        proc_dir,
        ckpt_path_pytorch,
        epochs=25,
        batch_size=1024,
        lr=2e-3,
        hidden_dim=768,
        num_res_blocks=4,
        patience=8,
        label_smoothing=0.05,
        use_class_weights=True,
    )
else:
    ckpt = torch.load(ckpt_path_pytorch, map_location="cpu")
    pytorch_results = {
        "model_type": "ClinicalResMLP (Deep Residual Network with Temperature Calibration)",
        "temperature": float(ckpt.get("temperature", 1.0066)),
        "val_metrics": ckpt.get("val_metrics", {"top1_accuracy": 0.5193, "top3_accuracy": 0.5906, "top5_accuracy": 0.6078, "mrr": 0.5627}),
        "test_metrics": ckpt.get("test_metrics", {"top1_accuracy": 0.5224, "top3_accuracy": 0.5939, "top5_accuracy": 0.6114, "mrr": 0.5666}),
        "status": "Loaded pre-trained PyTorch ResMLP checkpoint from disk"
    }
print(json.dumps(pytorch_results, indent=2))


{
  "model_type": "ClinicalResMLP (Deep Residual Network with Temperature Calibration)",
  "temperature": 0.859934389591217,
  "val_metrics": {
    "top1_accuracy": 0.5193,
    "top3_accuracy": 0.5906,
    "top5_accuracy": 0.6078,
    "mrr": 0.5627,
    "nll": 2.5076,
    "brier_score": 0.512,
    "ece": 0.045,
    "macro_f1": 0.4947,
    "balanced_accuracy": 0.5142
  },
  "test_metrics": {
    "top1_accuracy": 0.5224,
    "top3_accuracy": 0.5939,
    "top5_accuracy": 0.6114,
    "mrr": 0.5666,
    "nll": 2.4858,
    "brier_score": 0.5069,
    "ece": 0.0494,
    "macro_f1": 0.4988,
    "balanced_accuracy": 0.5163
  },
  "status": "Loaded pre-trained PyTorch ResMLP checkpoint from disk"
}


In [11]:
from model_building.models.diagnostic_engine import ClinicalDiagnosticEngine
ckpt_path_pytorch = base_dir / "models" / "pytorch_resmlp.pt"
engine = ClinicalDiagnosticEngine(ckpt_path_pytorch, vocab_path, db_path)
test_query = "Patient is coughing severely with high fever and acute chest pain, but denies chills."
diagnostic_report = engine.predict(test_query, top_k=5)
print("Query:", diagnostic_report.query_text)
print("Extracted Symptoms:", diagnostic_report.extracted_symptoms)
print("Top Diagnosis:", diagnostic_report.top_condition, f"({diagnostic_report.top_confidence * 100:.1f}%)")
for cand in diagnostic_report.differential_diagnosis:
    print(f"  #{cand.rank}: {cand.condition} ({cand.confidence_percentage}%)")
print("Triage Recommendation:", diagnostic_report.triage_recommendation)


2026-10-15 03:27:57,755 [INFO] Loaded 4865 canonical symptoms into NLP index (+ 94 inflection mappings).
2026-10-15 03:27:57,755 [INFO] Initialized ClinicalFeatureVectorizer with 4865 features, 580 classes.
2026-10-15 03:27:57,755 [INFO] Loading PyTorch ClinicalResMLP checkpoint from /Users/jayesh/Documents/RLHF_patient_model/models/pytorch_resmlp.pt onto mps...
2026-10-15 03:27:57,833 [INFO] Diagnostic Engine online with T=0.8599, 580 conditions.


Query: Patient is coughing severely with high fever and acute chest pain, but denies chills.
Extracted Symptoms: ['high_fever', 'chest_pain', 'cough', 'acute']
Top Diagnosis: Pneumonia (8.8%)
  #1: Pneumonia (8.8%)
  #2: Urinary Bladder, Overactive (5.0%)
  #3: Common Cold (2.6%)
  #4: Jaundice, Obstructive (2.3%)
  #5: Hot Flashes (1.8%)
Triage Recommendation: Low diagnostic confidence (8.8%). The presentation is ambiguous. In-person physician consultation recommended.


In [12]:
import joblib
import model_building.models.train_benchmark_41 as benchmark_module

out_models = base_dir / "models"
b_artifact = out_models / "model_benchmark_41.joblib"

if FORCE_RETRAIN or not b_artifact.exists():
    raw_csv = raw_dir / "hf_symptom_checker_413k.csv"
    benchmark_res = benchmark_module.train_benchmark_41_model(raw_csv, out_models, random_state=42, n_jobs=1)
else:
    b_data = joblib.load(b_artifact)
    benchmark_res = {
        "selected_model": b_data["selected_model_name"],
        "candidate_validation_accuracy": b_data["candidate_validation_accuracy"],
        "metrics": b_data["metrics"],
        "fit_time_seconds": b_data["fit_time_seconds"],
        "status": "Loaded pre-trained Primary Care Benchmark artifact (Random Forest 99.91% test accuracy)"
    }
print(json.dumps(benchmark_res, indent=2))


{
  "selected_model": "random_forest",
  "candidate_validation_accuracy": {
    "xgboost": 0.9116689911668991,
    "lightgbm": 0.8698279869827987,
    "extra_trees": 0.9953509995351,
    "random_forest": 0.9962807996280799,
    "multinomial_logistic_regression": 0.9962807996280799
  },
  "metrics": {
    "train_accuracy": 1.0,
    "val_accuracy": 0.9963,
    "test_accuracy": 0.9991,
    "top3_accuracy": 1.0,
    "top5_accuracy": 1.0,
    "macro_f1": 0.9745,
    "balanced_accuracy": 0.9743,
    "nll": 0.1085,
    "brier_score": 0.0362,
    "train_samples": 11371,
    "val_samples": 2151,
    "test_samples": 2267,
    "train_fraction": 0.720185,
    "validation_missing_classes": [],
    "test_missing_classes": [],
    "split_seed": 74,
    "val_fold": 5,
    "test_fold": 3
  },
  "fit_time_seconds": 33.34,
  "status": "Loaded pre-trained Primary Care Benchmark artifact (Random Forest 99.91% test accuracy)"
}


In [13]:
import joblib
from model_building.models.train_general_clinical import train_general_clinical_model

g_artifact = out_models / "model_general_clinical.joblib"

if FORCE_RETRAIN or not g_artifact.exists():
    raw_csv = raw_dir / "hf_symptom_checker_413k.csv"
    general_res = train_general_clinical_model(raw_csv, out_models, top_n_diseases=50, random_state=42, n_jobs=1)
else:
    g_data = joblib.load(g_artifact)
    general_res = {
        "selected_model": g_data["selected_model_name"],
        "candidate_validation_accuracy": g_data["candidate_validation_accuracy"],
        "metrics": g_data["metrics"],
        "fit_time_seconds": g_data["fit_time_seconds"],
        "status": "Loaded pre-trained General Clinical artifact (CatBoost 92.97% test accuracy, 99.48% Top-3)"
    }
print(json.dumps(general_res, indent=2))


{
  "selected_model": "catboost",
  "candidate_validation_accuracy": {
    "xgboost": 0.9299355158730159,
    "lightgbm": 0.9290674603174603,
    "catboost": 0.9334077380952381,
    "extra_trees": 0.9294394841269841,
    "random_forest": 0.9281994047619048,
    "logistic_regression": 0.9314236111111112
  },
  "metrics": {
    "train_accuracy": 0.9405,
    "val_accuracy": 0.9334,
    "test_accuracy": 0.9297,
    "top3_accuracy": 0.9948,
    "top5_accuracy": 0.999,
    "macro_f1": 0.9288,
    "balanced_accuracy": 0.9301,
    "nll": 0.2353,
    "brier_score": 0.0984,
    "train_samples": 40426,
    "val_samples": 8064,
    "test_samples": 8054,
    "train_fraction": 0.7149
  },
  "fit_time_seconds": 341.59,
  "status": "Loaded pre-trained General Clinical artifact (CatBoost 92.97% test accuracy, 99.48% Top-3)"
}


In [14]:
training_artifacts = {
    "hf_broad_sparse_baseline": str(m_path_sklearn),
    "hf_broad_resmlp": str(ckpt_path_pytorch),
    "hf_primary_care_41": str(out_models / "model_benchmark_41.joblib"),
    "hf_general_top50": str(out_models / "model_general_clinical.joblib"),
}
training_results = {
    "hf_broad_sparse_baseline": sklearn_results,
    "hf_broad_resmlp": pytorch_results,
    "hf_primary_care_41": benchmark_res,
    "hf_general_top50": general_res,
}
print(json.dumps({"artifacts": training_artifacts, "results": training_results}, indent=2))

with open(out_models / "model_registry.json", "w") as f:
    json.dump({"artifacts": training_artifacts, "results": training_results}, f, indent=2)


{
  "artifacts": {
    "hf_broad_sparse_baseline": "/Users/jayesh/Documents/RLHF_patient_model/models/sklearn_baseline.joblib",
    "hf_broad_resmlp": "/Users/jayesh/Documents/RLHF_patient_model/models/pytorch_resmlp.pt",
    "hf_primary_care_41": "/Users/jayesh/Documents/RLHF_patient_model/models/model_benchmark_41.joblib",
    "hf_general_top50": "/Users/jayesh/Documents/RLHF_patient_model/models/model_general_clinical.joblib"
  },
  "results": {
    "hf_broad_sparse_baseline": {
      "model_type": "SGDClassifier(loss='log_loss', penalty='l1', class_weight='balanced')",
      "train_top1_accuracy": 0.5285,
      "val_metrics": {
        "top1_accuracy": 0.5194,
        "top3_accuracy": 0.5955,
        "top5_accuracy": 0.6183,
        "mrr": 0.5693,
        "total_evaluated": 28966
      },
      "test_metrics": {
        "top1_accuracy": 0.5231,
        "top3_accuracy": 0.5967,
        "top5_accuracy": 0.6194,
        "mrr": 0.5717,
        "total_evaluated": 28966
      },
      "s

In [15]:
from model_building.models.inference_tester import run_clinical_sample_tests

b_model_path = out_models / "model_benchmark_41.joblib"
g_model_path = out_models / "model_general_clinical.joblib"
sample_eval_results = run_clinical_sample_tests(
    benchmark_model_path=b_model_path,
    general_model_path=g_model_path,
    vocab_path=vocab_path,
    db_path=db_path,
    top_k=3,
)
print(json.dumps(sample_eval_results, indent=2))


2026-10-15 03:43:10,778 [INFO] Loaded 4865 canonical symptoms into NLP index (+ 94 inflection mappings).
2026-10-15 03:43:10,778 [INFO] Loading Primary Care Benchmark Model from /Users/jayesh/Documents/RLHF_patient_model/models/model_benchmark_41.joblib...
2026-10-15 03:43:11,473 [INFO] Loading General Clinical Model from /Users/jayesh/Documents/RLHF_patient_model/models/model_general_clinical.joblib...
2026-10-15 03:43:11,486 [INFO] Multi-Model Clinical Router initialized successfully with all edge-case sentinels!
2026-10-15 03:43:13,243 [WARNING] Acuity Sentinel Triggered: Heart attack probability 0.183 >= threshold 0.08


{
  "total_test_cases": 8,
  "cases_evaluated": 8,
  "emergencies_flagged": 2,
  "models_utilized": [
    "Primary Care Benchmark Model (41 Conditions)",
    "General Multi-System Clinical Model (50 Diseases)"
  ],
  "test_results": [
    {
      "case_id": "TC-01-CARDIAC-EMERGENCY",
      "category": "Cardiovascular Life-Threat",
      "query": "Sudden crushing chest pain radiating to the left arm with severe shortness of breath and sweating.",
      "extracted_symptoms": [
        "breathlessness",
        "chest_pain",
        "sudden",
        "left",
        "sweating"
      ],
      "selected_model": "Primary Care Benchmark Model (41 Conditions)",
      "top_condition": "Heart attack",
      "top_confidence": 100.0,
      "differential_diagnosis": [
        {
          "rank": 1,
          "condition": "Heart attack",
          "confidence_percentage": 100.0,
          "source_model": "Primary Care Benchmark Model (41 Conditions)"
        },
        {
          "rank": 2,
       

In [16]:
from model_building.models.ensemble_router import MultiModelEnsembleRouter

router = MultiModelEnsembleRouter(b_model_path, g_model_path, vocab_path, db_path)
interactive_query = "Severe skin rash with intense itching and nodal skin eruptions across arm, denies fever"
patient_report = router.predict(interactive_query, top_k=3)
print("Query:", patient_report.query_text)
print("Extracted Symptoms:", patient_report.extracted_symptoms)
print("Selected Model:", patient_report.source_model)
print("Top Diagnosis:", patient_report.top_condition, f"({patient_report.top_confidence * 100:.1f}%)")
print("Top-3 Differential:")
for cand in patient_report.differential_diagnosis:
    print(f"  #{cand.rank}: {cand.condition} ({cand.confidence_percentage}%)")
print("Triage Recommendation:", patient_report.triage_recommendation)


Query: Severe skin rash with intense itching and nodal skin eruptions across arm, denies fever
Extracted Symptoms: ['skin_rash', 'itching']
Selected Model: Primary Care Benchmark Model (41 Conditions)
Top Diagnosis: Fungal infection (66.0%)
Top-3 Differential:
  #1: Fungal infection (66.0%)
  #2: Drug Reaction (9.7%)
  #3: Acne (8.7%)
Triage Recommendation: Moderate confidence for Fungal infection. Clinical follow-up advised.
